**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/malfatti-2026"], check=True)
    os.chdir("/content/encore/entries/malfatti-2026")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Malfatti, Velica et al. (2026) from dandiset 002056

Paper. Malfatti T, Velica A, et al. (2026). Increased layer 5 Martinotti cell excitation reduces
pyramidal cell population plasticity and improves learned motor execution. *eLife* 14:RP109286,
[doi:10.7554/eLife.109286](https://doi.org/10.7554/eLife.109286). The claims file was written from
the eLife reviewed preprint v3 (2026-11-02), its Methods, legends and supplement (Supplemental
Figures 1 to 7, Supplemental Tables 1 to 6); this attempt read the same text.

Dandiset. [DANDI:002056](https://dandiarchive.org/dandiset/002056/0.261113.1541), version
0.261113.1541. The paper cites no version of the dandiset. This is its only published version
(2026-11-13, "First release"), and the draft has the same 566 assets and size (95.86 GB), so the
published version is pinned.

Code. The paper's code, https://gitlab.com/malfatti/M1MartinottiOnMovement at commit 86efdf1
(the commit the claims file read), and the functions it calls from SciScripts
(https://gitlab.com/malfatti/SciScripts) at commit ef25f53, the last before 86efdf1. The prehension
detection and classification and the statistics follow them; `m1mc.py` says where each parameter
comes from.

What the dandiset holds. 224 NWB files of 31 mice in three groups, and 342 raw miniscope AVI
files (89.4 GB):

- the behaviour-only learning group (`BehaviourL5GqMC`), 13 mice, DeepLabCut (DLC) tracking of sessions 1 to 10;
- the miniscope group (`L5PCMSMCGq`), 7 mice, DLC tracking of sessions 1, 2, 5 and 6, and a
  `OnePhotonSeries` that points to the raw miniscope videos, with no extracted cells or traces;
- the LFP group (`LFPM1GqMC`), 11 mice, sessions 8 to 13, DLC tracking and one LFP channel at 30 kHz.

No histology, no caspase-ablation group and no pasta handling or hanging wire data are in it.

| Claim | Tested here |
|---|---|
| pc-selection-counts, supfig2i-baseline-lower, fig1j-session-effect-controls, fig1j-session2-success-lower, fig1j-genotype-session6, trial-adjacency, supfig3-baseline-epochs, supfig2j-phase-amplitudes, supfig2j-control-training-decrease, supfig2j-creplus-naive-failed, supfig4-ratio-interaction, supfig4-ratio-reconfiguration, fig2d-latency-accuracy, fig2d-control-temporal-separation, fig2d-genotype-latency, fig2i-peak-width, supfig6-width-training, assemblies-counts, fig3g-salience, fig3k-assembly-area, fig3l-resilience | Blocked, data missing: they need calcium traces and cells, which the dandiset does not hold; it has only 89.4 GB of raw video, and the paper does not give the CNMF-E parameters needed to extract them again (section "Imaging") |
| supfig1-cfos-validation, fig5b-ablation-efficiency | Blocked, data missing: histology |
| fig5d-pasta-drops, fig5gh-hanging-wire, fig5i-ablation-prehension | Blocked, data missing: the caspase group |
| fig1fh-no-genotype-effect-learning, fig1fh-session-effect | Tested, behaviour-only group |
| prehension-overlap | Tested, miniscope group DLC |
| fig4b-clz-improves-success | Tested, LFP group DLC |
| fig4d-theta-genotype, fig4e-high-theta-accuracy, fig4fh-gamma-session13, supfig7-lfp-nonprehension | Tested, LFP group |

In [ ]:
import resource
import time
import warnings

import numpy as np
import pandas as pd

import m1mc

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)

started = time.time()
assets = m1mc.list_assets()
files = m1mc.nwb_table(assets)
files["group"] = [p.split("/")[1] for p in files.path]
print(f"dandiset {m1mc.DANDISET_ID} version {m1mc.DANDISET_VERSION}: {len(assets)} assets, "
      f"{sum(a['size'] for a in assets) / 1e9:.2f} GB")

## Data check

Every file is read from the archive. Each check of the description against the values is
stated under its output.

In [ ]:
avi = [a for a in assets if a["path"].endswith(".avi")]
print(f"NWB files: {len(files)}; AVI files: {len(avi)} ({sum(a['size'] for a in avi) / 1e9:.1f} GB), "
      f"all under the miniscope subjects: {sorted({a['path'].split('/')[0] for a in avi})}")
print("\nNWB files by kind:", files.kind.value_counts().to_dict())
print("Size of NWB files by kind (GB):", (files.groupby("kind").size_mb.sum() / 1e3).round(2).to_dict())
print("\nSessions (as deposited) and trials per subject:")
print(files.groupby(["kind", "subject"]).apply(
    lambda g: " ".join(f"{s}" + (f"(t{t})" if t > 1 else "") for s, t in sorted(zip(g.session, g.trial))),
    include_groups=False).to_string())

In [ ]:
t = time.time()
beh_files = files[files.kind == "behavior"].sort_values(["subject", "session"]).reset_index(drop=True)
ms_files = files[files.kind == "ophys+behavior"].sort_values(["subject", "session", "trial"]).reset_index(drop=True)
lfp_files = files[files.kind == "ecephys+behavior"].sort_values(["subject", "session"]).reset_index(drop=True)
beh = m1mc.load_many(list(beh_files.asset), workers=16)
ms = m1mc.load_many(list(ms_files.asset), workers=16)
print(f"DLC of {len(beh)} behaviour-only and {len(ms)} miniscope files in {time.time() - t:.0f} s")
t = time.time()
lfp = m1mc.load_many(list(lfp_files.asset), lfp=True, workers=6)
print(f"DLC and LFP of {len(lfp)} LFP files in {time.time() - t:.0f} s; "
      f"peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

In [ ]:
def meta(fl, ds):
    return pd.DataFrame([dict(subject=d["subject"], session=r.session, trial=r.trial, genotype=d["genotype"],
                              sex=d["sex"], dlc_frames=d["X"].shape[0], dlc_rate=round(d["rate"], 3),
                              dlc_t0=d["t0"], dlc_s=round(d["X"].shape[0] / d["rate"], 1),
                              trial_s=round(float(d["trial_stop"][0] - d["trial_start"][0]), 1),
                              x_max=np.nanmax(d["X"][:, 5]), y_max=np.nanmax(d["Y"][:, 5]))
                         for (_, r), d in zip(fl.iterrows(), ds)])

mb, mm, ml = meta(beh_files, beh), meta(ms_files, ms), meta(lfp_files, lfp)
for name, m in [("behaviour-only", mb), ("miniscope", mm), ("LFP", ml)]:
    g = m.groupby("subject").genotype.agg(lambda x: sorted(set(x)))
    print(f"{name}: {m.subject.nunique()} subjects; genotypes {g.astype(str).value_counts().to_dict()}; "
          f"sex {m.sex.value_counts().to_dict()}")
    print(f"   DLC rate {sorted(m.dlc_rate.unique())} frames/s, DLC duration {m.dlc_s.min()} to {m.dlc_s.max()} s "
          f"(median {m.dlc_s.median()}), LPaw x up to {m.x_max.max():.0f} px, y up to {m.y_max.max():.0f} px")
print("\nFiles whose genotype differs from the subject's other files:")
print(mb[mb.genotype == "unknown"][["subject", "session", "genotype"]].to_string(index=False))
print("\nPaper: behaviour-only 13 mice (6 Cre+, 7 control); miniscope 6 analysed (3 and 3) of 8 implanted, "
      "male; LFP 11 (5 Cre+, 6 control), female. Camera 1920 x 1080 px at 50 frames/s (Methods).")
print("\nMiniscope files, DLC against imaging duration:")
print(pd.DataFrame([dict(subject=d["subject"], session=r.session, trial=r.trial, dlc_s=round(d["X"].shape[0] / d["rate"], 1),
                         imaging_frames=len(d["ms_timestamps"]), imaging_s=round(d["ms_timestamps"][-1], 1),
                         imaging_hz=round(1 / np.median(np.diff(d["ms_timestamps"])), 2), videos=len(d["ms_files"]))
                    for (_, r), d in zip(ms_files.iterrows(), ms)]).to_string(index=False))

The metadata agree with the paper on the counts of the behaviour-only group (13 mice, 6 Cre+ and
7 WT) and the LFP group (11 mice, 5 Cre+ and 6 WT). The miniscope group has 7 mice (4 WT, 3 Cre+)
where the paper analysed 6 (3 and 3), and one Cre+ mouse (67113) has sessions 1 and 2 only. Every
file says the mouse is female; the Methods say the miniscope mice were male. One file
(66415, session 6) gives the genotype as unknown, where the subject's other files say
ChRNA2-Cre. The DLC rate is the Methods' 50 frames/s, and positions stay inside a 1920 x 1080
frame. The miniscope session 1 of 67314 has 20 s of DLC tracking against 602 s of imaging, and
its session 2 has 455 s of imaging.

In [ ]:
d0 = lfp[0]
print(f"LFP: {sorted({len(d['lfp_channels']) for d in lfp})} channel per file, named {sorted({c for d in lfp for c in d['lfp_channels']})}; "
      f"rate {sorted({round(d['lfp_rate_in'], 3) for d in lfp})} Hz; stated unit {sorted({d['lfp_unit'] for d in lfp})}")
print(f"LFP start relative to the DLC start: {min(d['lfp_t0'] - d['t0'] for d in lfp):.3f} to "
      f"{max(d['lfp_t0'] - d['t0'] for d in lfp):.3f} s; LFP duration "
      f"{min(d['lfp_n_in'] / d['lfp_rate_in'] for d in lfp):.1f} to {max(d['lfp_n_in'] / d['lfp_rate_in'] for d in lfp):.1f} s")
sd = pd.DataFrame([dict(subject=d["subject"], genotype=d["genotype"], session=r.session, sd=d["lfp_raw_range"][2],
                        lo=d["lfp_raw_range"][0], hi=d["lfp_raw_range"][1]) for (_, r), d in zip(lfp_files.iterrows(), lfp)])
print("\nRaw LFP values (stated in volts), per mouse over its sessions:")
print(sd.groupby(["subject", "genotype"]).agg(sd_min=("sd", "min"), sd_max=("sd", "max"), lo=("lo", "min"), hi=("hi", "max")).round(1).to_string())

st = pd.DataFrame([dict(subject=d["subject"], session=r.session, start=d["start_time"]) for (_, r), d in zip(lfp_files.iterrows(), lfp)])
st["date"] = st.start.str[:10]
st["clock"] = st.start.str[11:19]
print("\nsession_start_time of the LFP files, per mouse (dates and times of day over its six sessions):")
print(st.groupby("subject").agg(dates=("date", lambda x: sorted(set(x))), times=("clock", lambda x: sorted(set(x)))).to_string())
print(f"files whose start time is 23:59:5x on the last day of a month: {int(st.clock.str.startswith('23:59:5').sum())} of {len(st)}")

The LFP is stated in volts, but its values have a standard deviation of 72 to 1057 and reach
thousands: they are microvolts, or raw amplifier units. Several mice reach the same bounds,
-6389.6 and 6389.6, which looks like the amplifier's range. Mouse 67584 has a standard deviation
of 4011 to 4397 in every session, 4 to 60 times the others, so one scale does not hold across
files. Every analysis below normalises power within a recording, which no scale changes.

The files' `session_start_time` is 23:59:5x on the last day of a month in all but one of the 66
LFP files (31 July, 31 August or 31 December 2021; one file of 69204 has 30 April 2020 at
18:26). These look like placeholders, not recording dates, so the files do not say when each
mouse was recorded. The mice are named below by their identifiers only.

In [ ]:
import hashlib
traces = [(r.path, d["Lh"][:, 5]) for fl, ds in [(beh_files, beh), (ms_files, ms), (lfp_files, lfp)]
          for (_, r), d in zip(fl.iterrows(), ds)]
by_hash = {}
for p, v in traces:
    by_hash.setdefault(hashlib.md5(np.round(v, 6).tobytes()).hexdigest(), []).append(p.split("/")[1])
same = [v for v in by_hash.values() if len(v) > 1]
print(f"{len(traces)} files; groups of files with identical LPaw likelihood traces: {len(same)}")
for v in same:
    print("   ", " = ".join(v))

print("\nMiniscope session 2 recorded twice (Trial1, Trial2):")
for m in ["67111", "67112"]:
    a = ms[ms_files.index[(ms_files.subject == m) & (ms_files.session == 2) & (ms_files.trial == 1)][0]]
    b = ms[ms_files.index[(ms_files.subject == m) & (ms_files.session == 2) & (ms_files.trial == 2)][0]]
    n = min(len(a["X"]), len(b["X"]))
    both = (a["Lh"][:n, 9] > 0.9) & (b["Lh"][:n, 9] > 0.9)
    r_lh = np.corrcoef(a["Lh"][:n, 5], b["Lh"][:n, 5])[0, 1]
    r_x = np.corrcoef(a["X"][:n, 9][both], b["X"][:n, 9][both])[0, 1]
    print(f"   {m}: DLC frames {len(a['X'])} and {len(b['X'])}; LPaw likelihood r = {r_lh:.3f}; pellet x where both "
          f"tracked ({both.sum()} frames) r = {r_x:.3f}, median difference {np.median(np.abs(a['X'][:n, 9][both] - b['X'][:n, 9][both])):.2f} px; "
          f"imaging frames {len(a['ms_timestamps'])} and {len(b['ms_timestamps'])}")

In every one of the 13 behaviour-only mice the deposited session 5 and session 6 hold the same
DLC trace: one recording deposited twice. The Methods say that session 5 of this group held both
halves ("After five minutes in the fifth session, the pellet was instead placed on a fixed
position on the plate ... for five more minutes"), so session 6 of the paper is the second half of
the deposited session 5, and the deposited session 6 is a copy. Each recording is counted once:
session 5 is split at 300 s and the deposited session 6 is dropped (helper `learning_table`).
The group then has 10 distinct sessions per mouse where the paper's statistics (F(10,110)) imply
11. The pellet's tracked position moves at 300 s in session 5 in all 13 mice (next cell, median
112 px), which supports the split.

The two session 2 recordings of 67111 have the same number of DLC frames and pellet positions
that agree frame by frame (r = 0.98, median 0.97 px apart): the same behaviour video tracked twice,
deposited with two different imaging recordings. Only Trial1 is used. Those of 67112 differ.

In [ ]:
rows = []
for (_, r), d in zip(beh_files.iterrows(), beh):
    if r.session != 5:
        continue
    t = np.arange(d["X"].shape[0]) / d["rate"]
    ok = d["Lh"][:, 9] > 0.7
    a, b = ok & (t < 300), ok & (t >= 300)
    rows.append(dict(mouse=d["subject"], dlc_t0=d["t0"], frames_before=int(a.sum()), frames_after=int(b.sum()),
                     y_before=np.median(d["Y"][a, 9]) if a.any() else np.nan, y_after=np.median(d["Y"][b, 9]) if b.any() else np.nan,
                     x_before=np.median(d["X"][a, 9]) if a.any() else np.nan, x_after=np.median(d["X"][b, 9]) if b.any() else np.nan))
s5 = pd.DataFrame(rows).round(0)
s5["shift_px"] = np.hypot(s5.x_after - s5.x_before, s5.y_after - s5.y_before).round(0)
print("Deposited session 5: median tracked pellet position (likelihood > 0.7) before and after 300 s")
print(s5.to_string(index=False))
print(f"DLC start time of the session 5 files: {sorted(s5.dlc_t0.unique())} s")
print(f"\nMedian shift {s5.shift_px.median():.0f} px; mice with a shift over 50 px: {(s5.shift_px > 50).sum()} of {s5.shift_px.notna().sum()}")

## Prehension detection

Prehensions are detected and classified as the paper's code does (helper docstring). The code's
onset detector (`QuantifyTTLs` with its default threshold of mean + 2.5 SD of the binary trace)
finds every onset only while less than about 13.8% of frames are in a prehension block; above that
it would find none. The next cell shows the fraction in every file, and compares the two edge
rules and the two sets of body parts (the code's LPaw and LFinger1, the Methods' paw and three
fingers).

In [ ]:
rows = []
for name, fl, ds in [("behaviour-only", beh_files, beh), ("miniscope", ms_files, ms), ("LFP", lfp_files, lfp)]:
    for (_, r), d in zip(fl.iterrows(), ds):
        p = m1mc.prehensions(d)
        pa = m1mc.prehensions(d, edge="any")
        p4 = m1mc.prehensions(d, m1mc.LEFT_ALL)
        rows.append(dict(group=name, frac_on=p.attrs["frac_frames_on"], n=len(p), n_any=len(pa), n_all4=len(p4),
                         invalid=p.attrs["n_invalid"], success=int(p.success.sum()), **{c: int((p.cls == c).sum()) for c in
                         ["Success", "HitMiss", "Miss", "HitDrop", "NoPellet"]}))
det = pd.DataFrame(rows)
print(det.groupby("group").agg(files=("n", "size"), max_frac_on=("frac_on", "max"), prehensions=("n", "sum"),
                               same_with_any_edge=("n", lambda x: int((x.values == det.loc[x.index, "n_any"].values).sum())),
                               paw_and_3_fingers=("n_all4", "sum"), at_video_edge=("invalid", "sum"),
                               files_without_prehension=("n", lambda x: int((x == 0).sum()))).to_string())
print("\nClasses (Core.ReachClassify), all files:")
print(det[["Success", "HitMiss", "Miss", "HitDrop", "NoPellet"]].sum().to_string())
print(f"\nSuccess ratio over all prehensions: {det.success.sum() / det.n.sum():.3f}")

No file comes near 13.8% of frames in a prehension block, so the code's threshold finds every
onset and both edge rules give the same prehensions in every file. The Methods' four body parts
find fewer prehensions; they are shown as a variant where a claim uses counts. Six of the 28
miniscope files have no prehension at all.

In [ ]:
L = []
for (_, r), d in zip(ms_files.iterrows(), ms):
    p = m1mc.prehensions(d)
    lh = d["Lh"]
    dur = d["X"].shape[0] / d["rate"]
    L.append(dict(mouse=d["subject"], genotype=d["genotype"], session=r.session, trial=r.trial, dlc_s=round(dur, 1),
                  lpaw=(lh[:, 5] > 0.7).mean(), lfinger1=(lh[:, 6] > 0.7).mean(),
                  both=((lh[:, 5] > 0.7) & (lh[:, 6] > 0.7)).mean(), pellet=(lh[:, 9] > 0.7).mean(),
                  prehensions=len(p), per_10min=len(p) / (dur / 600), success=int(p.success.sum())))
msd = pd.DataFrame(L)
print("Miniscope group, per file: fraction of frames with likelihood > 0.7 (LPaw, LFinger1, both, pellet) and prehensions")
print(msd.round(3).to_string(index=False))
print("\nPrehensions per genotype and session (sum over mice; Trial2 of 67111 session 2 is a copy and left out):")
x = msd[~((msd.mouse == "67111") & (msd.session == 2) & (msd.trial == 2))]
print(x.pivot_table(index="genotype", columns="session", values="prehensions", aggfunc="sum").to_string())
print("Mice with at least one prehension, per genotype and session:")
print(x.assign(any=x.prehensions > 0).pivot_table(index="genotype", columns="session", values="any", aggfunc="sum").to_string())
print(f"Share of all prehensions from each mouse: {(x.groupby('mouse').prehensions.sum() / x.prehensions.sum()).round(3).to_dict()}")

cmp = []
for name, fl, ds in [("behaviour-only, sessions 1, 2, 5", beh_files, beh), ("LFP, sessions 8 to 13", lfp_files, lfp)]:
    for (_, r), d in zip(fl.iterrows(), ds):
        if name.startswith("behaviour") and r.session not in (1, 2, 5):
            continue
        lh = d["Lh"]
        cmp.append(dict(group=name, both=((lh[:, 5] > 0.7) & (lh[:, 6] > 0.7)).mean(),
                        per_10min=len(m1mc.prehensions(d)) / (d["X"].shape[0] / d["rate"] / 600)))
cmp = pd.concat([pd.DataFrame(cmp), x.assign(group="miniscope, sessions 1, 2, 5, 6")[["group", "both", "per_10min"]]])
print("\nFraction of frames with LPaw and LFinger1 above 0.7, and prehensions per 10 min, per file (median and range):")
print(cmp.groupby("group").agg(files=("both", "size"), both_median=("both", "median"), both_min=("both", "min"), both_max=("both", "max"),
                               per_10min_median=("per_10min", "median"), per_10min_min=("per_10min", "min"),
                               per_10min_max=("per_10min", "max")).round(3).to_string())


In the miniscope files the left paw and its first finger are both above 0.7 likelihood in a
median of 0.2% of frames (0 to 3.3%), against 1.4% in sessions 1, 2 and 5 of the behaviour-only
group and 2.4% in the LFP group. The median miniscope file has 9.7 prehensions per 10 min,
against 66.8 and 117.2. LFinger1 is the body part that is rarely tracked: it is above 0.7 in at
most 3.6% of frames of a miniscope file, while LPaw is above 0.7 in up to 25.7% (81% in the 20 s
file of 67314). Four mice give 97% of the miniscope prehensions (67109 38%, 67114 21%, 67112 21%
and 67111 17%). The other three give 21 (67110), 2 (67113) and 4 (67314) over all their
sessions. Cre+ mice give 6 and 23 prehensions in sessions 1 and 2, from 2 and 3 mice. The paper
analysed PCs around prehensions in 3 Cre+ and 3 control mice in all four sessions. So the
prehensions detected here in the miniscope group may well not match the authors'. The paper does
not say which 6 mice it used.

## Imaging

The miniscope NWB files hold the DLC tracking and a `OnePhotonSeries` whose data are empty and
whose `external_file` lists the raw AVI videos. No ROI, cell, fluorescence or trace data are in any
file (next cell). All 21 imaging claims need the CNMF-E traces of the paper's pipeline and the
matching of cells across sessions. The Methods name the steps ("Miniscope videos were spatially
downsampled 3x and motion-corrected, then cells signal was extracted using ... CNMF-E ...
Neuron identity was matched along multiple recordings through intersection over union metric and
the Hungarian algorithm", "Neural signal extraction") but not the CNMF-E parameters (neuron size,
correlation and peak-to-noise thresholds, merging) or the matching threshold, and the paper's
code at commit 86efdf1 starts from the extracted traces. The cell also measures the archive's
streaming rate, to show the cost of extracting the traces again.


In [ ]:
import h5py
with m1mc.open_h5(ms_files.asset.iloc[0]) as h:
    o = h["acquisition/OnePhotonSeries"]
    print("OnePhotonSeries data shape:", o["data"].shape, "; external files:", len(o["external_file"]))
    print("processing modules:", list(h["processing"].keys()), "->", list(h["processing/behavior"].keys())[:4], "...")
    print("ophys processing module present:", "ophys" in h["processing"])
kinds = set()
for (_, r) in ms_files.iterrows():
    with m1mc.open_h5(r.asset) as h:
        kinds |= set(h["processing"].keys())
print("processing modules over all miniscope files:", sorted(kinds))

import remfile
f = remfile.File(m1mc.asset_url(avi[0]))
t = time.time(); n = 0
while n < 100e6:
    b = f.read(10_000_000)
    if not b:
        break
    n += len(b)
rate = n / (time.time() - t)
vol = sum(a["size"] for a in avi)
print(f"\nstreamed {n / 1e6:.0f} MB of {avi[0]['path']} at {rate / 1e6:.0f} MB/s")
print(f"the {len(avi)} videos ({vol / 1e9:.1f} GB) would take {vol / rate / 60:.0f} min to stream alone, before decoding, "
      f"motion correction and CNMF-E of {len(ms_files)} sessions of about 12000 frames each")

Streaming the videos alone takes tens of minutes at the rate measured in the run (printed
above); decoding them, motion correction and CNMF-E
of the 28 recordings, then cell registration across sessions, would take hours more on this
runner's 4 CPUs. The deciding fact, though, is that the traces and cells the claims rest on are
not deposited, and the parameters to derive them again are not given. The imaging claims are
blocked as `data_missing`.


## Clock of the LFP and the tracking

The LFP and the DLC tracking have their own start times in each file. The protocol asks to show
that they share a clock. The 1-30 Hz LFP is averaged at the prehension onsets shifted by every
lag from -60 to 60 s, and the strength of the locked potential is scored at each lag (helper
`locked_lag`, `pooled_lag`). If the streams share a clock the potential is strongest near lag 0.

In [ ]:
lagtab = []
for (_, r), d in zip(lfp_files.iterrows(), lfp):
    L, sc, n = m1mc.locked_lag(d)
    far = np.abs(L) > 5
    i0 = np.argmin(np.abs(L))
    lagtab.append(dict(mouse=d["subject"], genotype=d["genotype"], session=r.session, prehensions=n,
                       best_lag_s=round(float(L[np.argmax(sc)]), 2),
                       z_best=round(float((sc.max() - sc[far].mean()) / sc[far].std()), 1),
                       z_at_0=round(float((sc[i0] - sc[far].mean()) / sc[far].std()), 1)))
lagtab = pd.DataFrame(lagtab)
print(lagtab.pivot_table(index=["mouse", "genotype"], columns="session", values="best_lag_s").to_string())
print(lagtab.pivot_table(index=["mouse", "genotype"], columns="session", values="z_at_0").to_string())
pooled = m1mc.pooled_lag(lfp)
pooled = pd.DataFrame(pooled).T.round(2)
print("\nPer mouse, the score curve averaged over its six sessions:")
print(pooled.to_string())
SHIFT = {m: float(v.best_lag) for m, v in pooled.iterrows() if v.z_at_0 < 2}
weak = lagtab[~lagtab.mouse.isin(list(SHIFT)) & (lagtab.z_at_0 < 2)]
print("Sessions of the other mice without a locked potential at lag 0 (z < 2), with the best lag and its z:")
print(weak[["mouse", "session", "prehensions", "best_lag_s", "z_best", "z_at_0"]].to_string(index=False))
print("\nMice with no locked potential at the deposited alignment (z at lag 0 < 2), and their lag:", SHIFT)

In 7 of the 11 mice a potential locked to prehension onset is present at lag 0: their pooled
curves peak within 0.6 s of 0, except 66228, whose pooled peak at -9.8 s comes from session 8
alone while its sessions 11 to 13 peak within 0.5 s of 0. The sessions of these mice without a
locked potential at lag 0 are listed above; all but two (66128 session 11 and 66546 session 12)
are saline sessions, which the LFP claims do not use. Those two have their strongest locked
potential away from lag 0 (66128 session 11 at 3.3 s, z 4.5; 66546 session 12 at -2.0 s, z
18.6), so their clock could not be established. The four mice 69107, 69109, 69110 and 69204 (all WT) show no locked potential at lag
0 in any session (z -1.4 to 1.9), and in 16 of their 24 sessions it appears 6.9 to 9.0 s before
the DLC onsets (pooled lags -8.0 to -8.7 s). This is consistent with their LFP being on a clock
about 8 s off the tracking, but it does not show it: the files hold no LED or sync trace, and the
missing potential could have other causes.

The LFP claims are judged on the files as deposited, the data as the dandiset gives them and the
reading that needs no choice of ours. Two variants are shown. In one, the four mice are left out
(4 Cre+ and 2 WT remain among the mice with both accuracies). In the other, their onsets are moved
by the lag found (`SHIFT`). The realigned variant is not used for any outcome, for two reasons.
It is circular, since the lag is chosen to maximise the prehension-locked LFP and the claims
then test prehension-locked power on the same data. It is also confounded with genotype, since
only WT mice are moved.


## fig1fh-no-genotype-effect-learning

> In a separate group of mice given CLZ before every session (6 Cre+, 7 controls), genotype had no
> effect on the total number of prehensions, the number of successful prehensions or the success
> ratio.

Criterion: a mixed ANOVA, genotype between and session within (ANOVA-type statistics when Shapiro's
test rejects normality), on per-mouse, per-session values; reproduced when the genotype effect has
p > 0.05 for all three measures. Readings: (a) all sessions, or sessions 1, 2 and 6 as
BehavOnly.py tests; (b) raw counts, or counts per 10 min as the code does. The deposited session 5
is split at 300 s (Methods) and the copy in session 6 dropped. Variants: the copy kept as an
eleventh session (the paper's degrees of freedom), the Methods' four body parts, and each half of
session 5 divided by its own length. The paper's code divides both halves by the whole
recording's duration (BehavOnly.py lines 333-344 copy its TotalTimes to both, line 378 divides by
it), so its counts per 10 min of sessions 5 and 6 are per 10 min of the recording, not of the half.

In [ ]:
lt = m1mc.learning_table(beh_files, beh)
lt_copy = m1mc.learning_table(beh_files, beh, keep_copy=True)
lt_4 = m1mc.learning_table(beh_files, beh, parts=m1mc.LEFT_ALL)
lt_h = m1mc.learning_table(beh_files, beh, halves="own")
print("Mice per genotype:", lt.groupby("gen").mouse.nunique().to_dict(), "; sessions:", sorted(lt.session.unique()))
print("\nMeans over mice (raw counts per session, success ratio):")
print(lt.groupby(["session", "gen"])[["total_raw", "success_raw", "ratio"]].mean().unstack().round(3).to_string())
print("\nBy mouse, total prehensions per session:")
print(lt.pivot_table(index=["gen", "mouse"], columns="session", values="total_raw").to_string())

MEASURES = {"total prehensions": ("total", "total_raw"), "successful prehensions": ("success", "success_raw"),
            "success ratio": ("ratio", "ratio")}
res = []
for tname, tab in [("main", lt), ("copy kept as a session", lt_copy), ("paw and 3 fingers", lt_4),
                    ("halves per own length", lt_h)]:
    for sname, sess in [("all sessions", None), ("sessions 1, 2, 6", [1, 2, 6])]:
        d = tab if sess is None else tab[tab.session.isin(sess)]
        for meas, (norm, raw) in MEASURES.items():
            for cname, col in ([("per 10 min", norm), ("raw", raw)] if norm != raw else [("ratio", norm)]):
                a = m1mc.auto_anova(d, col, "mouse", "gen", ["session"])
                ma = m1mc.mixed_anova(d, col, "mouse", "gen", ["session"])
                at = m1mc.ats(d, col, "mouse", "gen", ["session"])
                g, s = a.set_index("effect").loc["gen"], a.set_index("effect").loc["session"]
                gm = d.groupby(["gen", "mouse"])[col].mean().groupby("gen").mean()
                res.append(dict(table=tname, sessions=sname, measure=meas, counts=cname, test=a.attrs["test"],
                                gen_F=round(g.F, 3), gen_df=f"{g.df1:.0f},{g.df2:.0f}" if np.isfinite(g.df2) else f"{g.df1:.2f},inf",
                                gen_p=g.p, session_F=round(s.F, 3), session_df=f"{s.df1:.0f},{s.df2:.0f}" if np.isfinite(s.df2) else f"{s.df1:.2f},inf",
                                session_p=s.p, cre_minus_wt=round(gm["ChRNA2-Cre"] - gm["WT"], 3),
                                gen_p_anova=ma.set_index("effect").p["gen"], gen_p_ats=at.set_index("effect").p["gen"],
                                session_p_anova=ma.set_index("effect").p["session"], session_p_ats=at.set_index("effect").p["session"]))
learn = pd.DataFrame(res)
show = learn.copy()
for c in ["gen_p", "session_p", "gen_p_anova", "gen_p_ats", "session_p_anova", "session_p_ats"]:
    show[c] = show[c].map(lambda x: f"{x:.3g}")
print("\nPaper (Supplemental Table 1A-F): genotype F(1,11) = 0.048, 0.034, 0.336 (p 0.83, 0.856, 0.574); "
      "session F(10,110) = 9.525, 7.571, 5.536 (p 2.9e-11, 4.3e-09, 1.2e-06)")
print(show.drop(columns=["session_F", "session_df", "session_p", "session_p_anova", "session_p_ats"]).to_string(index=False))
print(f"\nSmallest genotype p over all readings and variants: {learn.gen_p.min():.3f} (main table: "
      f"{learn[learn.table == 'main'].gen_p.min():.3f}); under the mixed ANOVA {learn.gen_p_anova.min():.3f}, the ATS {learn.gen_p_ats.min():.3f}")

**Result: reproduced.** Under every reading the genotype effect has p > 0.05 for all three
measures: the smallest is 0.145 (success ratio, sessions 1, 2 and 6), and over all sessions the
genotype p values are 0.84, 0.84 and 0.58 (paper 0.83, 0.856 and 0.574). The variants give the
same answer (smallest p 0.141); with each half of session 5 divided by its own length the genotype
p values per 10 min are 0.94, 0.94, 0.83 and 0.58.

## fig1fh-session-effect

> Mice improved in the task, with an effect of training session on total prehensions, successful
> prehensions and success ratio; totals and successes rose from session 1 to 2, fell in the harder
> sessions 5 and 6, and were stable after session 7.

Criterion: the same test over all sessions recorded; reproduced when the session effect has
p < 0.05 for all three measures and the mean over mice of total and successful prehensions is
higher in session 2 than in session 1 and lower in sessions 5 and 6 than in session 2.

In [ ]:
print("Paper: session F(10,110) = 9.525 (total), 7.571 (successful), 5.536 (success ratio)")
print(show[show.sessions == "all sessions"][["table", "measure", "counts", "test", "session_F", "session_df", "session_p",
                                             "session_p_anova", "session_p_ats"]].to_string(index=False))
for tname, tab in [("main", lt), ("copy kept as a session", lt_copy), ("paw and 3 fingers", lt_4),
                    ("halves per own length", lt_h)]:
    for col in ["total", "total_raw", "success", "success_raw"]:
        m = tab.groupby(["session", "mouse"])[col].mean().groupby("session").mean()
        print(f"{tname:24s} {col:12s} means over mice: s1 {m[1]:.1f}, s2 {m[2]:.1f}, s5 {m[5]:.1f}, s6 {m[6]:.1f}; "
              f"s2 > s1 {m[2] > m[1]}, s5 < s2 {m[5] < m[2]}, s6 < s2 {m[6] < m[2]}; "
              f"later sessions: {', '.join(f's{s:g} {m[s]:.1f}' for s in sorted(m.index) if s >= 7)}")

**Result: reproduced.** The session effect has p < 0.05 for all three measures in every reading,
with F = 9.57, 7.86 and 6.30 against the paper's 9.525, 7.571 and 5.536. The means over mice of
totals and successes rise from session 1 to 2 and fall in sessions 5 and 6 below session 2 in raw
counts (total 48.1, 90.0, 61.5 and 44.8; successful 16.7, 24.3, 15.2 and 11.2) and in counts per
10 min as the paper's code computes them (47.4, 88.7, 60.8 and 44.3; 16.5, 24.0, 15.0 and 11.0).
The code's values for sessions 5 and 6 are per 10 min of the whole recording, not of the half.
With each half divided by its own length, session 5 has more prehensions per 10 min than session
2 (123.1 against 88.7 total, 30.3 against 24.0 successful) and session 6 slightly fewer (87.5 and
21.8), so under that variant the decrease in session 5 does not hold; its session effects still
have p of 0.001 or less. Figure 1F-H plots the number of prehensions, which the raw counts give.
The session effect has 9 and 99 degrees of freedom here, against the paper's 10 and 110, as one
recording per mouse is a copy; with the copy counted, the degrees of freedom are the paper's. With
the sessions after 5 renumbered +1, as the paper's code does, sessions 8 to 11 average 141.5 to
177.6 total prehensions per 10 min, above every one of sessions 1 to 6; "stable after session 7"
is not judged.

## prehension-overlap

> Some prehensions fell within 2 s of another prehension, more often for failed than successful
> ones; 3.17% and 36.07% in session 1, 6.08% and 28.26% in session 2, 1.79% and 28.12% in session
> 5, 4.76% and 29.71% in session 6, and 4.0% and 29.91% over all sessions.

Criterion: the miniscope group, sessions 1, 2, 5 and 6, pooled over mice and genotypes; readings
(a) the next prehension starts within 2 s after the onset, (b) the previous one started within 2
s before, (c) either. Reproduced when, under one reading, at least 8 of the 10 values are within
tolerance and failed exceeds successful in every session; partially reproduced when failed exceeds
successful in every session but fewer values match. The paper does not say which mice; all 7 are
used, with 67113 (two sessions only) and 67314 (20 s of tracking in session 1) left out as
variants.

In [ ]:
EXPECT = {1: (3.17, 36.07), 2: (6.08, 28.26), 5: (1.79, 28.12), 6: (4.76, 29.71), "all": (4.0, 29.91)}
rows = []
for (_, r), d in zip(ms_files.iterrows(), ms):
    if r.subject == "67111" and r.session == 2 and r.trial == 2:
        continue  # the same behaviour video as Trial1 (data check)
    for pname, parts in [("code", m1mc.LEFT_CODE), ("paw and 3 fingers", m1mc.LEFT_ALL)]:
        p = m1mc.prehensions(d, parts)
        a, b = m1mc.overlap_flags(p.t.values)
        for i in range(len(p)):
            rows.append(dict(mouse=r.subject, session=r.session, parts=pname, success=bool(p.success.iloc[i]),
                             a=bool(a[i]), b=bool(b[i]), c=bool(a[i] or b[i])))
ov = pd.DataFrame(rows)
x = ov[ov.parts == "code"]
print("Prehensions per mouse and session (code detection):")
print(x.groupby(["mouse", "session"]).size().unstack().fillna(0).astype(int).to_string())
print("Success ratio per mouse and session:")
print(x.groupby(["mouse", "session"]).success.mean().unstack().round(2).to_string())

out = []
for pname in ["code", "paw and 3 fingers"]:
    for excl in [(), ("67113",), ("67314",), ("67113", "67314"), ("67109",)]:
        y = ov[(ov.parts == pname) & ~ov.mouse.isin(excl)]
        for rd in "abc":
            vals, match = {}, 0
            for s in [1, 2, 5, 6, "all"]:
                z = y if s == "all" else y[y.session == s]
                sv, fv = 100 * z[z.success][rd].mean(), 100 * z[~z.success][rd].mean()
                es, ef = EXPECT[s]
                match += (abs(sv - es) <= 0.1 * es + 1e-9) + (abs(fv - ef) <= 0.1 * ef + 1e-9)
                vals[f"S{s}"], vals[f"F{s}"] = round(sv, 2), round(fv, 2)
            f_gt_s = all(vals[f"F{s}"] > vals[f"S{s}"] for s in [1, 2, 5, 6])
            out.append(dict(parts=pname, left_out=",".join(excl) or "-", reading=rd, within_tol=match,
                            failed_gt_success_all_sessions=f_gt_s, n=len(y), **vals))
ovt = pd.DataFrame(out)
print("\nPaper: S1 3.17 F1 36.07, S2 6.08 F2 28.26, S5 1.79 F5 28.12, S6 4.76 F6 29.71, all S 4.0 F 29.91 (%)")
print(ovt.to_string(index=False))
print("\nOverlap per mouse, reading (a), code detection (% of the mouse's prehensions of each accuracy; n successful/failed):")
x = ov[ov.parts == "code"]
pm = x.groupby(["mouse", "session", "success"]).agg(pct=("a", lambda v: 100 * v.mean()), n=("a", "size")).unstack("success")
print(pm.round(1).to_string())
mm = x.groupby(["mouse", "session", "success"]).a.mean().mul(100).groupby(["session", "success"]).mean().unstack()
print("Mean over mice of the per-mouse percentage, reading (a) (mice with prehensions of that accuracy):")
print(mm.round(2).rename(columns={False: "failed", True: "successful"}).to_string())
print(f"\nMost values within tolerance under any reading: {ovt.within_tol.max()} of 10; failed above successful in "
      f"every session under {int(ovt.failed_gt_success_all_sessions.sum())} of {len(ovt)} readings and variants")
print(f"Successful prehensions overlapping, over all readings and variants: {ovt[[f'S{s}' for s in [1, 2, 5, 6]]].min().min():.1f} to {ovt[[f'S{s}' for s in [1, 2, 5, 6]]].max().max():.1f}%")

**Result: partially reproduced.** Failed prehensions overlap with another more often than
successful ones in every session under every reading with all seven mice (reading (a): 47.6,
42.6, 48.4 and 44.0% against 13.9, 15.5, 10.7 and 29.7%), and under 27 of the 30 readings and
variants. The three exceptions are the four-body-part detection with 67109 left out, where neither
accuracy overlaps in session 1 (0.0 and 0.0%). At most three of the ten percentages are within
tolerance under any reading or variant (that variant, reading (b)), and at most one with all
seven mice. The miniscope tracking finds few prehensions (data check), so the magnitudes may reflect a
detection that differs from the paper's in this group.

## fig4b-clz-improves-success

> In mice trained before hM3Dq expression, CLZ (sessions 11-13) raised the prehension success rate
> of Cre+ mice compared with saline (sessions 8-10), which was not seen in controls.

Criterion: the LFP group, 5 Cre+ and 6 controls, success ratio per mouse and session; mixed ANOVA
on genotype and session, then per genotype ANOVA on session and on treatment, with two readings of
the treatment test: (a) within mouse on per-mouse saline and CLZ means; (b) one-way ANOVA over the
mouse-session ratios. Reproduced when the Cre+ treatment effect has p < 0.05 with CLZ higher and the
control effect p > 0.05. The criterion names sessions 7 to 13; session 7 is not in the dandiset, so
sessions 8 to 13 are used.

In [ ]:
from scipy import stats
rows = []
for (_, r), d in zip(lfp_files.iterrows(), lfp):
    for pname, parts in [("code", m1mc.LEFT_CODE), ("paw and 3 fingers", m1mc.LEFT_ALL)]:
        p = m1mc.prehensions(d, parts)
        rows.append(dict(parts=pname, mouse=d["subject"], gen=d["genotype"], session=r.session, total=len(p),
                         success=int(p.success.sum())))
sr = pd.DataFrame(rows)
sr["ratio"] = sr.success / sr.total.where(sr.total > 0)
sr["treat"] = np.where(sr.session >= 11, "CLZ", "NaCl")
print("Success ratio per mouse and session (code detection):")
print(sr[sr.parts == "code"].pivot_table(index=["gen", "mouse"], columns="session", values="ratio").round(3).to_string())
print("Prehensions:")
print(sr[sr.parts == "code"].pivot_table(index=["gen", "mouse"], columns="session", values="total").to_string())
for pname in ["code", "paw and 3 fingers"]:
    d = sr[(sr.parts == pname)].dropna(subset=["ratio"])
    a = m1mc.auto_anova(d, "ratio", "mouse", "gen", ["session"])
    print(f"\n[{pname}] genotype x session, {a.attrs['test']} ({a.attrs['n_subjects']} mice):\n{m1mc.fmt(a)}")
    for g in ["ChRNA2-Cre", "WT"]:
        x = d[d.gen == g]
        s = m1mc.auto_anova(x, "ratio", "mouse", None, ["session"])
        m = x.groupby(["mouse", "treat"]).ratio.mean().unstack()
        tt = stats.ttest_rel(m.CLZ, m.NaCl)
        ow = stats.f_oneway(x[x.treat == "CLZ"].ratio, x[x.treat == "NaCl"].ratio)
        kw = stats.kruskal(x[x.treat == "CLZ"].ratio, x[x.treat == "NaCl"].ratio)
        print(f"[{pname}] {g}: session {s.attrs['test']} F({s.df1[0]:.0f},{s.df2[0]:.0f}) = {s.F[0]:.3f}, p = {s.p[0]:.3g}; "
              f"mean ratio NaCl {m.NaCl.mean():.3f}, CLZ {m.CLZ.mean():.3f}, CLZ higher in {(m.CLZ > m.NaCl).sum()} of {len(m)} mice")
        print(f"      (a) within mouse F(1,{len(m) - 1}) = {tt.statistic ** 2:.3f}, p = {tt.pvalue:.3g}; "
              f"(b) one-way F(1,{len(x) - 2}) = {ow.statistic:.3f}, p = {ow.pvalue:.3g}; Kruskal-Wallis p = {kw.pvalue:.3g}")

**Result: reproduced.** In Cre+ mice the success ratio is higher under CLZ than saline (0.135
against 0.075, in 5 of 5 mice), at p = 0.011 within mouse and p = 0.0085 one-way, and in controls
neither reading finds an effect (p = 0.62 and 0.63). The session effect is significant in Cre+
mice (p = 0.00045) and not in controls (p = 0.90), as the paper reports. The Methods' four body
parts give the same answer.

## LFP band power

Band power as the Methods describe (helper `band_power`): second-order Butterworth band-pass,
spectrogram, normalisation by the mean, mean over the bins in the band; mean power over the 1 s
before (phase "before") and the 1 s after onset ("during"), averaged over the prehensions of each
accuracy per mouse and session, sessions 11 to 13. Readings, all run: spectrogram window 1 s or
0.5 s (Hann, 50% overlap); normalisation per frequency bin over the recording (`bin`), over all
bins (`all`) or by the running mean of each bin (`cum`, the literal "cumulative mean"); and the
alignment: as deposited (the reading the outcomes use), with the four mice without a locked
potential at lag 0 left out, or with their onsets moved by the lag found (circular, shown only
for comparison; see the clock check). The file holds one channel per
mouse ("All accepted MOp5 channels"), so reading (b) of the claims file, the channel, has one
answer. Mice with prehensions of both accuracies in all three sessions are used.

The before and during windows take the spectrogram bins whose centres fall in them. With 1 s
windows a bin centred just before onset covers up to 0.5 s after it, so the two phases overlap
in time; the 0.5 s windows halve this. No judged contrast is between the phases.


In [ ]:
t = time.time()
READINGS = [(w, n) for w in (1.0, 0.5) for n in ("bin", "all", "cum")]
rows, outside = [], []
for (_, r), d in zip(lfp_files.iterrows(), lfp):
    if r.session < 11:
        continue
    p = m1mc.prehensions(d)
    onsets = {"deposited": p.t.values, "realigned": p.t.values + SHIFT.get(d["subject"], 0.0)}
    for band, bf in m1mc.BANDS.items():
        for win, norm in READINGS:
            tt, pw = m1mc.band_power(d["lfp"], d["lfp_rate"], bf, win, norm)
            tt = tt + d["lfp_t0"]
            for align, on in onsets.items():
                b, a = m1mc.window_means(tt, pw, on)
                for acc in (True, False):
                    m = p.success.values == acc
                    for phase, v in (("before", b), ("during", a)):
                        rows.append(dict(alignment=align, win=win, normalisation=norm, band=band, mouse=d["subject"], gen=d["genotype"],
                                         session=r.session, acc="success" if acc else "failed", phase=phase,
                                         n=int(m.sum()), power=np.nanmean(v[m]) if m.any() else np.nan))
                outside.append(dict(alignment=align, win=win, normalisation=norm, band=band, mouse=d["subject"], gen=d["genotype"],
                                    session=r.session, power=m1mc.outside_mean(tt, pw, on)))
bp, bo = pd.DataFrame(rows), pd.DataFrame(outside)
OFFSET_MICE = sorted(SHIFT)
bp = pd.concat([bp, bp[(bp.alignment == "deposited") & ~bp.mouse.isin(OFFSET_MICE)].assign(alignment="four left out")], ignore_index=True)
bo = pd.concat([bo, bo[(bo.alignment == "deposited") & ~bo.mouse.isin(OFFSET_MICE)].assign(alignment="four left out")], ignore_index=True)
cnt = bp[(bp.band == "full") & (bp.win == 1.0) & (bp.normalisation == "bin") & (bp.alignment == "deposited") & (bp.phase == "before")]
print(f"band power in {time.time() - t:.0f} s")
print("Prehensions per mouse, session and accuracy, sessions 11 to 13:")
print(cnt.pivot_table(index=["gen", "mouse"], columns=["session", "acc"], values="n").to_string())
complete = cnt.groupby("mouse").apply(lambda g: (g.n > 0).all() and len(g) == 6, include_groups=False)
KEEP = sorted(complete[complete].index)
print(f"\nmice with both accuracies in all three sessions: {len(KEEP)} ({sorted(set(cnt.mouse) - set(KEEP))} left out); "
      f"genotypes {bp[bp.mouse.isin(KEEP)].groupby('gen').mouse.nunique().to_dict()}")
print(f"power ranges, all readings: around prehensions {bp.power.min():.3g} to {bp.power.max():.3g}, "
      f"outside {bo.power.min():.3g} to {bo.power.max():.3g}")


def lfp_tests(band, model, effects, session=None):
    # The auto test (and both families) per reading, for the effects named.
    out = []
    for (align, win, norm), g in bp[bp.mouse.isin(KEEP) & (bp.band == band)].groupby(["alignment", "win", "normalisation"]):
        d = g if session is None else g[g.session == session]
        within = model if session is None else [w for w in model if w != "session"]
        a = m1mc.auto_anova(d, "power", "mouse", "gen", within)
        ma = m1mc.mixed_anova(d, "power", "mouse", "gen", within).set_index("effect")
        at = m1mc.ats(d, "power", "mouse", "gen", within).set_index("effect")
        row = dict(alignment=align, win=win, normalisation=norm, test=a.attrs["test"][:9])
        a = a.set_index("effect")
        for e in effects:
            row[f"{e} F"] = round(a.F[e], 3)
            row[f"{e} p"] = round(a.p[e], 4)
            row[f"{e} p ANOVA/ATS"] = f"{ma.p[e]:.3f}/{at.p[e]:.3f}"
        row["Cre+ mean"] = round(d[d.gen != "WT"].power.mean(), 4)
        row["WT mean"] = round(d[d.gen == "WT"].power.mean(), 4)
        row["success mean"] = round(d[d.acc == "success"].power.mean(), 4)
        row["failed mean"] = round(d[d.acc == "failed"].power.mean(), 4)
        out.append(row)
    return pd.DataFrame(out)

## fig4d-theta-genotype

> Around prehensions, LFP power in the full theta band (4-12 Hz) and the low theta band (4-7 Hz)
> was higher in Cre+ mice than in controls, and theta power differed with accuracy.

Criterion: a mixed ANOVA (ANOVA-type when not normal) on genotype, accuracy, phase and session,
mice as observations; reproduced when the genotype effects on theta and low theta have p < 0.05
with Cre+ higher.

In [ ]:
MODEL = ["acc", "phase", "session"]
print("Paper: theta genotype F(1,8) = 6.783, p = 0.031; accuracy F(1,8) = 6.332, p = 0.036; low theta genotype F(1,8) = 6.489, p = 0.034")
th = lfp_tests("theta", MODEL, ["gen", "acc"])
lt_ = lfp_tests("low_theta", MODEL, ["gen", "acc"])
print("\ntheta (4-12 Hz)"); print(th.to_string(index=False))
print("\nlow theta (4-7 Hz)"); print(lt_.to_string(index=False))
ok = (th["gen p"] < 0.05) & (lt_["gen p"] < 0.05) & (th["Cre+ mean"] > th["WT mean"]) & (lt_["Cre+ mean"] > lt_["WT mean"])
print("\nper alignment, readings with both genotype effects at p < 0.05 and Cre+ higher:", ok.groupby(th.alignment).sum().to_dict(),
      "; theta genotype p range:", th.groupby("alignment")["gen p"].agg(["min", "max"]).round(3).to_dict("index"),
      "; low theta:", lt_.groupby("alignment")["gen p"].agg(["min", "max"]).round(3).to_dict("index"))
print("theta accuracy p, per alignment:", th.groupby("alignment")["acc p"].agg(["min", "max"]).round(3).to_dict("index"))
print(f"\nreadings where both genotype effects have p < 0.05 with Cre+ higher: {int(ok.sum())} of {len(ok)}; "
      f"theta genotype p {th['gen p'].min():.3f} to {th['gen p'].max():.3f}, low theta {lt_['gen p'].min():.3f} to {lt_['gen p'].max():.3f}; "
      f"Cre+ above WT in {int((th['Cre+ mean'] > th['WT mean']).sum())} and {int((lt_['Cre+ mean'] > lt_['WT mean']).sum())} of {len(th)}")

**Result: not reproduced.** With the files as deposited, no reading gives a genotype effect at
p < 0.05 on theta (p 0.235 to 0.480) or low theta (p 0.198 to 0.447) under the test the
criterion names, though Cre+ mean power is above control in all six readings. With the four mice
left out the genotype p is 0.76 or more, and with them realigned 0.14 or more. The accuracy
effect on theta is not significant in any reading (p 0.18 to 0.51). Shapiro's test selects the
mixed ANOVA in all six deposited readings. The ANOVA-type statistic alone would give a theta
genotype p below 0.05 in four of them (0.017 to 0.034), so the outcome rests on the criterion's
choice of test.

## fig4e-high-theta-accuracy

> In the high theta band (8-12 Hz), LFP power was higher on successful than on failed prehensions.

Criterion: band power and readings as in fig4d-theta-genotype, high theta band; reproduced when
the accuracy effect has p < 0.05 with successful higher.

In [ ]:
print("Paper: accuracy F(1,8) = 8.724, p = 0.018")
ht = lfp_tests("high_theta", MODEL, ["acc", "gen"])
print(ht.to_string(index=False))
ok = (ht["acc p"] < 0.05) & (ht["success mean"] > ht["failed mean"])
print("per alignment: readings with the effect", ok.groupby(ht.alignment).sum().to_dict(), "; accuracy p range",
      ht.groupby("alignment")["acc p"].agg(["min", "max"]).round(3).to_dict("index"))
print(f"\nreadings with the accuracy effect at p < 0.05 and successful higher: {int(ok.sum())} of {len(ok)}; "
      f"accuracy p {ht['acc p'].min():.3f} to {ht['acc p'].max():.3f}; successful above failed in "
      f"{int((ht['success mean'] > ht['failed mean']).sum())} of {len(ht)}")

**Result: not reproduced.** Successful prehensions have higher mean high theta power than failed
ones in all 18 readings, but the accuracy effect is not significant in any: p 0.211 to 0.482 as
deposited, and 0.117 to 0.745 over all readings and variants.

## fig4fh-gamma-session13

> Gamma band effects appeared only in session 13, where power was higher on successful than failed
> prehensions, and Cre+ mice had higher low gamma (30-40 Hz) and high gamma (60-90 Hz) power than
> controls.

Criterion: band power and readings as in fig4d-theta-genotype, session 13 tested alone (genotype,
accuracy and phase); reproduced when in session 13 the genotype effects on low and high gamma have
p < 0.05 with Cre+ higher, and successful exceeds failed in at least one gamma band at p < 0.05;
partially reproduced when the genotype part holds for one band.

In [ ]:
print("Paper: session 13 low gamma genotype F(1,inf) = 8.781, p = 0.0091; high gamma genotype F(1,8) = 9.329, p = 0.048")
g13 = {b: lfp_tests(b, MODEL, ["gen", "acc"], session=13) for b in ["gamma", "low_gamma", "high_gamma"]}
for b, t in g13.items():
    print(f"\n{b}, session 13"); print(t.to_string(index=False))
lo, hi = g13["low_gamma"], g13["high_gamma"]
gen_lo = (lo["gen p"] < 0.05) & (lo["Cre+ mean"] > lo["WT mean"])
gen_hi = (hi["gen p"] < 0.05) & (hi["Cre+ mean"] > hi["WT mean"])
acc_any = sum((t["acc p"] < 0.05) & (t["success mean"] > t["failed mean"]) for t in g13.values()) > 0
print(f"\nreadings with: low gamma genotype {int(gen_lo.sum())}, high gamma genotype {int(gen_hi.sum())}, both "
      f"{int((gen_lo & gen_hi).sum())}, an accuracy effect in a gamma band {int(acc_any.sum())}, everything {int((gen_lo & gen_hi & acc_any).sum())} "
      f"(of {len(lo)})")
print("per alignment, readings with low gamma genotype:", gen_lo.groupby(lo.alignment).sum().to_dict(),
      "; high gamma genotype:", gen_hi.groupby(hi.alignment).sum().to_dict())
for b, t in g13.items():
    print(f"{b} genotype p per alignment:", t.groupby("alignment")["gen p"].agg(["min", "max"]).round(3).to_dict("index"),
          "; accuracy p:", t.groupby("alignment")["acc p"].agg(["min", "max"]).round(3).to_dict("index"))
print("Under which readings one genotype part holds:")
print(pd.DataFrame(dict(alignment=lo.alignment, win=lo.win, normalisation=lo.normalisation, low=gen_lo, high=gen_hi)).to_string(index=False))
print("\nOver all three sessions, for comparison:")
for b in ["gamma", "low_gamma", "high_gamma"]:
    t = lfp_tests(b, MODEL, ["gen", "acc"])
    print(f"{b}: genotype p {t['gen p'].min():.3f} to {t['gen p'].max():.3f}; accuracy p {t['acc p'].min():.3f} to {t['acc p'].max():.3f}")

**Result: not reproduced.** In session 13, with the files as deposited, neither gamma genotype
effect holds in any reading (low gamma p 0.173 to 0.400, high gamma 0.140 to 0.194), and no
gamma band has an accuracy effect (p 0.138 to 0.976). With the four mice left out, neither
genotype effect holds (p 0.384 or more), though the full gamma band has an accuracy
effect in two readings (p 0.016 and 0.049). Only the realigned variant gives one genotype part, in
four of six readings (low gamma in three, high gamma in one), and it is circular and confounded
with genotype (clock check). The full gamma band has a genotype effect as deposited in three
readings (ATS, p 0.005 to 0.011), but the criterion names the low and high gamma bands.

## supfig7-lfp-nonprehension

> Outside prehensions, genotype and session interacted in all frequency bands; in Cre+ mice full,
> theta and beta power fell in session 13, low theta power depended on genotype and session, low and
> high gamma on genotype, and high theta showed no effect.

Criterion: band power averaged per mouse and session over the time outside every -1 to 1 s
prehension window, sessions 11 to 13, beta 13-30 Hz; mixed ANOVA on genotype and session per band.
Reproduced when the interaction has p < 0.05 in at least five of the seven plotted bands, Cre+
session 13 power is below sessions 11 and 12 for full, theta and beta, and high theta shows no
effect at p < 0.05; partially reproduced when the interaction holds in three or four bands. All
11 mice are used.

In [ ]:
print("Paper: interaction F(2,18) > 3.584, p < 0.049 in each band; Cre+ session effect for full, theta, beta F(2,8) > 8.667")
SEVEN = ["full", "theta", "beta", "gamma", "low_theta", "low_gamma", "high_gamma"]
out = []
for (align, win, norm), g in bo.groupby(["alignment", "win", "normalisation"]):
    row = dict(alignment=align, win=win, normalisation=norm)
    n_int = 0
    for band in SEVEN + ["high_theta"]:
        d = g[g.band == band]
        a = m1mc.auto_anova(d, "power", "mouse", "gen", ["session"]).set_index("effect")
        if band in SEVEN:
            n_int += a.p["gen:session"] < 0.05
            row[f"{band} int p"] = round(a.p["gen:session"], 3)
        else:
            row["high_theta min p"] = round(a.p.min(), 3)
    cre = g[g.gen != "WT"].groupby(["band", "session"]).power.mean().unstack()
    row["Cre+ s13 lowest (full, theta, beta)"] = all(cre.loc[b, 13] < min(cre.loc[b, 11], cre.loc[b, 12]) for b in ["full", "theta", "beta"])
    row["bands with interaction"] = int(n_int)
    out.append(row)
s7 = pd.DataFrame(out)
print(s7.to_string(index=False))
print(f"\nbands with a genotype x session interaction at p < 0.05: {s7['bands with interaction'].min()} to "
      f"{s7['bands with interaction'].max()} of 7 over the readings")
print("per alignment:", s7.groupby("alignment")["bands with interaction"].agg(["min", "max"]).to_dict("index"))
x = bo[(bo.alignment == "deposited") & (bo.win == 1.0) & (bo.normalisation == "bin")]
print("\nMeans per genotype and session, deposited, 1 s, per-bin normalisation:")
print(x.groupby(["band", "gen", "session"]).power.mean().unstack().round(3).to_string())

**Result: not reproduced.** With the files as deposited, the genotype by session interaction has
p < 0.05 in none of the seven bands in any of the six readings. With the four mice left out it
holds in at most one band, and with them realigned in at most two. Cre+ power in session 13 is
below sessions 11 and 12 for full, theta and beta in four of the six deposited readings (not
with the running mean), for example 0.894 against 1.002 and 0.988 (full, per-bin, 1 s), and high
theta has no effect at p < 0.05 as deposited (smallest p 0.137). The interaction the claim
rests on is absent.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| fig1fh-no-genotype-effect-learning | Reproduced |
| fig1fh-session-effect | Reproduced |
| prehension-overlap | Partially reproduced |
| fig4b-clz-improves-success | Reproduced |
| fig4d-theta-genotype | Not reproduced |
| fig4e-high-theta-accuracy | Not reproduced |
| fig4fh-gamma-session13 | Not reproduced |
| supfig7-lfp-nonprehension | Not reproduced |
| 21 imaging claims | Blocked: data missing (no traces or cells, raw video only) |
| supfig1-cfos-validation, fig5b-ablation-efficiency | Blocked: data missing (histology) |
| fig5d-pasta-drops, fig5gh-hanging-wire, fig5i-ablation-prehension | Blocked: data missing (caspase group) |

How the analysis differs from the paper's. Prehensions and their accuracy come from the
paper's code applied to the deposited DLC tracking; the paper's own per-prehension tables are not
public, so the detection cannot be compared one by one. The behaviour-only group has 10 distinct
sessions per mouse where the paper has 11; session 5 is split at the Methods' five minutes, as the
code's per-mouse split times are not public, and its halves are counted per 10 min of the whole
recording, as the code does; per 10 min of each half, session 5 would not fall below session 2. The LFP group lacks session 7. The LFP is decimated
to 1 kHz before filtering, and the spectrogram window, the normalisation and the channel are not
given by the paper; every reading is shown. The LFP claims are judged on the files as
deposited; leaving out the four mice without a locked potential at lag 0 gives the same outcomes.
Realigning them (circular, shown only for comparison) gives one genotype part of
fig4fh-gamma-session13 in four of six readings, the criterion's partial outcome, and changes none
of the other outcomes. The
miniscope tracking detects far fewer prehensions than that of the other groups. The ANOVA-type statistic is a Python implementation of
the nparLD formulas, not R's nparLD.

What was found about the dataset. The miniscope data are raw video only, with no cells or
traces, so none of the imaging findings can be tested without rerunning the extraction. The
behaviour-only group's session 6 is a copy of session 5 in all 13 mice. The two session 2
recordings of miniscope mouse 67111 hold the same behaviour video. The LFP of four control mice shows
no prehension-locked potential at the deposited alignment and one about 8 s earlier, consistent
with an offset that the files cannot confirm. The LFP files' session start times look like placeholders (23:59:5x on the last day of a
month). The LFP is labelled volts
but holds values in the hundreds to thousands, with one mouse at a scale 4 to 60 times the
others. All mice are labelled female, against the paper's male miniscope mice, and one file
gives a genotype as unknown. The histology and caspase-ablation data are not in the dandiset.

In [ ]:
print(f"notebook ran in {(time.time() - started) / 60:.1f} min; peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")